We decided to use spaCy and VADER.

In [ ]:
from nltk.sentiment import vader
from nltk.sentiment.vader import SentimentIntensityAnalyzer
vader_model = SentimentIntensityAnalyzer()

def vader_output_to_label(vader_output):
    """
    map vader output e.g.,
    {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound': 0.4215}
    to one of the following values:
    a) positive float -> 'positive'
    b) 0.0 -> 'neutral'
    c) negative float -> 'negative'
    
    :param dict vader_output: output dict from vader
    
    :rtype: str
    :return: 'negative' | 'neutral' | 'positive'
    """
    compound = vader_output['compound']
    
    if compound < 0:
        return 'negative'
    elif compound == 0.0:
        return 'neutral'
    elif compound > 0.0:
        return 'positive'
    
assert vader_output_to_label( {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound': 0.0}) == 'neutral'
assert vader_output_to_label( {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound': 0.01}) == 'positive'
assert vader_output_to_label( {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound': -0.01}) == 'negative'

In [1]:
import pandas as pd
import spacy
from spacy.tokens import Doc
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

nlp = spacy.load("en_core_web_sm")
vader = SentimentIntensityAnalyzer()

ModuleNotFoundError: No module named 'vaderSentiment'

In [4]:
ner_df = pd.read_csv("NER-test.tsv", sep="\t")
st_df  = pd.read_csv("Sentiment-topic-test.tsv", sep="\t")

sentences = []
for sid, grp in ner_df.groupby("sentence id"):
    grp = grp.sort_values("token id")
    meta = st_df[st_df["sentence id"] == sid].iloc[0]
    sentences.append({
        "sid": int(sid),
        "tokens": grp["token"].tolist(),
        "gold_bio": grp["BIO NER tag"].tolist(),
        "text": meta["text"],
        "gold_sentiment": meta["sentiment"],
        "gold_topic": meta["topic"],
    })
print(len(sentences), "sentences loaded")
sentences[0]

10 sentences loaded


{'sid': 0,
 'tokens': ['It',
  'took',
  'eight',
  'years',
  'for',
  'Warner',
  'Brothers',
  'to',
  'recover',
  'from',
  'the',
  'disaster',
  'that',
  'was',
  'this',
  'movie',
  '.'],
 'gold_bio': ['O',
  'O',
  'O',
  'O',
  'O',
  'B-ORG',
  'I-ORG',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 'text': 'It took eight years for Warner Brothers to recover from the disaster that was this movie.',
 'gold_sentiment': 'negative',
 'gold_topic': 'movie'}

In [5]:
SPACY2GOLD = {
    "PERSON":"PER", "ORG":"ORG",
    "GPE":"LOC", "LOC":"LOC", "FAC":"LOC",
    "NORP":"MISC", "LANGUAGE":"MISC", "WORK_OF_ART":"MISC",
    "EVENT":"MISC", "PRODUCT":"MISC", "LAW":"MISC",
}  # unmapped types (DATE, MONEY, CARDINAL, ...) -> O

def spacy_bio(tokens):
    doc = Doc(nlp.vocab, words=tokens)        # force gold tokenization
    for _, pipe in nlp.pipeline:
        doc = pipe(doc)
    out = []
    for tok in doc:
        if tok.ent_iob_ == "O":
            out.append("O")
        else:
            m = SPACY2GOLD.get(tok.ent_type_)
            out.append(f"{tok.ent_iob_}-{m}" if m else "O")
    return out

s = sentences[0]
for t, g, p in zip(s["tokens"], s["gold_bio"], spacy_bio(s["tokens"])):
    print(f"{t:14} gold={g:8} pred={p:8}{'' if g==p else '   <-- diff'}")

It             gold=O        pred=O       
took           gold=O        pred=O       
eight          gold=O        pred=O       
years          gold=O        pred=O       
for            gold=O        pred=O       
Warner         gold=B-ORG    pred=B-ORG   
Brothers       gold=I-ORG    pred=I-ORG   
to             gold=O        pred=O       
recover        gold=O        pred=O       
from           gold=O        pred=O       
the            gold=O        pred=O       
disaster       gold=O        pred=O       
that           gold=O        pred=O       
was            gold=O        pred=O       
this           gold=O        pred=O       
movie          gold=O        pred=O       
.              gold=O        pred=O       


In [6]:
def vader_label(text):
    c = vader.polarity_scores(text)["compound"]
    return "positive" if c >= 0.05 else "negative" if c <= -0.05 else "neutral"

for s in sentences:
    p = vader_label(s["text"])
    print(f'{s["sid"]}  gold={s["gold_sentiment"]:8} pred={p:8}'
          f'{"" if p==s["gold_sentiment"] else "  <-- diff"}  {s["text"][:50]}')

0  gold=negative pred=negative  It took eight years for Warner Brothers to recover
1  gold=positive pred=positive  All the New York University students love this din
2  gold=negative pred=positive  <-- diff  This Italian place is really trendy but they have 
3  gold=positive pred=positive  In conclusion, my review of this book would be: I 
4  gold=neutral  pred=positive  <-- diff  The story of this movie is focused on Carl Brashea
5  gold=neutral  pred=positive  <-- diff  Chris O'Donnell stated that while filming for this
6  gold=positive pred=positive  My husband and I moved to Amsterdam 6 years ago an
7  gold=positive pred=positive  Dame Maggie Smith performed her role excellently, 
8  gold=neutral  pred=neutral   The new movie by Mr. Kruno was shot in New York, b
9  gold=negative pred=positive  <-- diff  I always have loved English novels, but I just cou
